In [62]:
import pandas as pd
import numpy as np
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, callbacks,utils
from imblearn.over_sampling import ADASYN
from collections import Counter
from keras.optimizers import Adam

from sklearn.utils import class_weight
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE


In [63]:
df=pd.read_csv("cleaned_dataset_taiwan_2months.csv")
df.head()

,date,sitename,county,aqi,status,so2,co,o3,pm10,pm2.5,no2,nox,no,siteid
0,2024-08-31 23:00:00,Hukou,Hsinchu County,62.0,Moderate,0.9,0.17,35.0,18.0,17.0,2.3,2.6,0.3,22
1,2024-08-31 23:00:00,Zhongming,Taichung City,50.0,Good,1.6,0.32,27.9,27.0,14.0,7.6,9.3,1.6,31
2,2024-08-31 23:00:00,Zhudong,Hsinchu County,45.0,Good,0.4,0.17,25.1,21.0,13.0,2.9,4.1,1.1,23
3,2024-08-31 23:00:00,Hsinchu,Hsinchu City,42.0,Good,0.8,0.20,30.0,19.0,10.0,4.0,4.8,0.7,24
4,2024-08-31 23:00:00,Toufen,Miaoli County,50.0,Good,1.0,0.16,33.5,18.0,14.0,1.8,3.1,1.2,25


In [64]:
features=['so2','co','o3','pm2.5','pm10','no2','nox','no']
status=['status']
X=df[features]
y=df[status]  


In [65]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
y_encoded = utils.to_categorical(y_encoded, num_classes=num_classes)

c:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [66]:
X_train,X_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=17)

In [67]:
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [68]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X_train_res)

In [69]:
model = tf.keras.Sequential([
# Input shape: (Time Steps, Features)
    layers.Input(shape=(24, 8)), 
    
    # First GRU layer 
    layers.GRU(64, return_sequences=True),
    layers.BatchNormalization(), # Added for stability based on your previous design
    layers.Dropout(0.3),
    
    # Second GRU layer
    layers.GRU(32),
    layers.BatchNormalization(),
    layers.Dropout(0.2),


    # Output Layer
    layers.Dense(num_classes, activation='softmax')
])

In [70]:
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [71]:
# callback for saving checkpoint and early stop
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='loss', 
    factor=0.2, 
    patience=3, 
    min_lr=1e-6
)

my_callbacks = [
    callbacks.EarlyStopping(monitor='loss', patience=7, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath='best_model_rf.keras', monitor='accuracy', save_best_only=True),
    reduce_lr
]

In [72]:
weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(np.argmax(y_train_res, axis=1)),
    y=np.argmax(y_train_res, axis=1)
)
class_weights_dict = dict(enumerate(weights))

In [73]:
def create_sequences(data, labels, window_size=24):
    X, y = [], []
    for i in range(len(data) - window_size):
        # Extract the window of 24 hours
        X.append(data[i:(i + window_size)])
        # The target is the air quality status at the end of that window
        y.append(labels[i + window_size])
    return np.array(X), np.array(y)

# Transform your balanced, scaled data into 3D sequences
X_train_seq, y_train_seq = create_sequences(X_scaled, y_train_res, window_size=24)

In [74]:
model.fit(
X_train_seq, y_train_seq,
epochs=20,
batch_size=128,
class_weight=class_weights_dict,
shuffle=True,
validation_split=0.2,
verbose=1,
callbacks=my_callbacks
)#training nn

Epoch 1/20
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 72s 39ms/step - accuracy: 0.9502 - loss: 0.1814 - precision: 0.9522 - recall: 0.9485 - val_accuracy: 1.0000 - val_loss: 4.3412e-07 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 2/20
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 65s 38ms/step - accuracy: 0.9600 - loss: 0.1451 - precision: 0.9600 - recall: 0.9600 - val_accuracy: 1.0000 - val_loss: 3.2271e-07 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 3/20
1724/1724 ━━━━━━━━━━━━━━━━━━━━ 60s 35ms/step - accuracy: 0.9600 - loss: 0.1431 - precision: 0.9600 - recall: 0.9600 - val_accuracy: 1.0000 - val_loss: 4.7675e-07 - val_precision: 1.0000 - val_recall: 1.0000 - learning_rate: 5.0000e-04
Epoch 4/20
 335/1724 ━━━━━━━━━━━━━━━━━━━━ 44s 32ms/step - accuracy: 0.9598 - loss: 0.1436 - precision: 0.9598 - recall: 0.9598

KeyboardInterrupt: 

In [75]:
import numpy as np

# 1. Generate pseudo-labels from the GRU model
# Note: These are based on the sequences (window size of 24)
nn_pseudo_labels = model.predict(X_train_seq)

y_pseudo_train = np.argmax(nn_pseudo_labels, axis=1)

# 2. Align your X data
# We must skip the first 24 rows of the original X_scaled 
# to match the length of the sequences/pseudo-labels
X_train_aligned = X_scaled[24:] 

# 3. Train the Random Forest on the aligned data
rf_distilled = RandomForestClassifier(n_estimators=100, random_state=20)
rf_distilled.fit(X_train_aligned, y_pseudo_train)

# 4. Evaluation
# Ensure X_test is scaled using the same scaler before predicting
X_test_scaled = scaler.transform(X_test)
y_pred_rf = rf_distilled.predict(X_test_scaled)

print("--- Distilled Random Forest Performance ---")
y_test_integers = np.argmax(y_test, axis=1)
print(classification_report(y_test_integers, y_pred_rf, target_names=le.classes_))

8619/8619 ━━━━━━━━━━━━━━━━━━━━ 56s 6ms/step
--- Distilled Random Forest Performance ---
                                precision    recall  f1-score   support

                          Good       0.94      0.99      0.96     22923
                      Moderate       0.71      0.35      0.47      2230
Unhealthy for Sensitive Groups       0.31      0.13      0.19        30

                      accuracy                           0.93     25183
                     macro avg       0.65      0.49      0.54     25183
                  weighted avg       0.92      0.93      0.92     25183



In [76]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Convert y_test from a 2D one-hot matrix to 1D integer labels
y_test_ground_truth = np.argmax(y_test, axis=1)

# Now both variables are 1D arrays of integers
accuracy = accuracy_score(y_test_ground_truth, y_pred_rf)
precision, recall, f1, _ = precision_recall_fscore_support(y_test_ground_truth, y_pred_rf, average='weighted')


print("--- Overall Categorical Performance (Global) ---")
print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")


--- Overall Categorical Performance (Global) ---
Accuracy:  92.90%
Precision: 0.9183
Recall:    0.9290
F1-Score:  0.9177


In [77]:
import joblib

# 1. Save the trained Random Forest model
#joblib.dump(rf_model, 'rf_aqi_classifier.pkl')

# 2. Save the LabelEncoder (Crucial for decoding results later)
#joblib.dump(le, 'aqi_label_encoder.pkl')